In [1]:
import duckdb
import pandas as pd
# Charger l'extension SQL magique
# %load_ext sql
%reload_ext sql

# Établir la connexion avec DuckDB
# Connexion en mémoire (temporaire)
%sql duckdb:///

# Ou connexion persistante (optionnel)
# %sql duckdb:///ma_base.duckdb
# Configuration complète de JupySQL
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = 1  # Affiche le nombre de lignes et le temps
%config SqlMagic.displaycon = False

Connecting to 'duckdb:///'

In [2]:
%%sql
-- Créer des tables à partir des fichiers CSV
CREATE OR REPLACE TABLE pizza_sales AS 
SELECT * FROM read_csv_auto('pizza_sales.csv');

-- Vérifier que les tables sont créées
SHOW TABLES;

,Success


In [ ]:
%%sql
# Afficher les données de la table Pizza_sales
select * from Pizza_sales ;

,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1,1,hawaiian_m,1,2022-01-01,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2,2,classic_dlx_m,1,2022-01-01,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3,2,five_cheese_l,1,2022-01-01,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4,2,ital_supr_l,1,2022-01-01,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5,2,mexicana_m,1,2022-01-01,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza
...,...,...,...,...,...,...,...,...,...,...,...,...
48615,48616,21348,ckn_alfredo_m,1,2022-12-31,21:23:10,16.75,16.75,M,Chicken,"Chicken, Red Onions, Red Peppers, Mushrooms, A...",The Chicken Alfredo Pizza
48616,48617,21348,four_cheese_l,1,2022-12-31,21:23:10,17.95,17.95,L,Veggie,"Ricotta Cheese, Gorgonzola Piccante Cheese, Mo...",The Four Cheese Pizza
48617,48618,21348,napolitana_s,1,2022-12-31,21:23:10,12.00,12.00,S,Classic,"Tomatoes, Anchovies, Green Olives, Red Onions,...",The Napolitana Pizza
48618,48619,21349,mexicana_l,1,2022-12-31,22:09:54,20.25,20.25,L,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


In [4]:
%%sql
-- Le revenu total des ventes
SELECT SUM(total_price) AS somme
FROM pizza_sales;

,somme
0,817860.05


In [5]:
%%sql
-- Le Nombre de pizza vendu 
SELECT count( DISTINCT order_id) AS Nombre_vendu
FROM pizza_sales;

,Nombre_vendu
0,21350


In [6]:
%%sql
-- total de vente par mois
SELECT MONTH(order_date) AS Mois, SUM(total_price) AS somme
FROM pizza_sales
GROUP BY Mois;

,Mois,somme
0,1,69793.30
1,2,65159.60
2,3,70397.10
3,4,68736.80
4,5,71402.75
5,6,68230.20
6,7,72557.90
7,8,68278.25
8,9,64180.05
9,10,64027.60


In [7]:
%%sql
-- total de vente par mois et par nom du pizza
SELECT MONTH(order_date) AS Mois,pizza_name , SUM(total_price) AS somme
FROM pizza_sales
GROUP BY Mois,pizza_name;

,Mois,pizza_name,somme
0,1,The Hawaiian Pizza,2442.75
1,1,The Classic Deluxe Pizza,2941.50
2,1,The Barbecue Chicken Pizza,3770.25
3,1,The Spinach Supreme Pizza,1357.00
4,1,The Spicy Italian Pizza,2762.00
...,...,...,...
379,12,The Italian Capocollo Pizza,2122.00
380,12,The Spinach Pesto Pizza,1169.25
381,12,The Pepperoni Pizza,2293.00
382,12,The Greek Pizza,2109.45


In [8]:
%%sql
-- total de vente par pizza_category
SELECT pizza_category , SUM(total_price) AS somme
FROM pizza_sales
GROUP BY pizza_category;

,pizza_category,somme
0,Classic,220053.10
1,Supreme,208197.00
2,Chicken,195919.50
3,Veggie,193690.45


In [9]:
%%sql
-- total de vente par la taille du pizza
SELECT pizza_size, SUM(total_price) AS somme
FROM pizza_sales
GROUP BY pizza_size;

,pizza_size,somme
0,S,178076.50
1,XL,14076.00
2,M,249382.25
3,L,375318.70
4,XXL,1006.60


In [10]:
%%sql
-- le nombre de vente de pizza par mois
SELECT MONTH(order_date) AS Mois ,COUNT(DISTINCT pizza_id) AS nombre
FROM pizza_sales
GROUP BY Mois;

,Mois,nombre
0,2,3892
1,8,4094
2,9,3819
3,1,4156
4,6,4025
5,10,3797
6,12,3859
7,4,4067
8,5,4239
9,7,4301


In [11]:
%%sql
-- le nombre de vente de pizza par categorrie
SELECT pizza_category ,COUNT(DISTINCT pizza_id) AS nombre
FROM pizza_sales
GROUP BY  pizza_category;

,pizza_category,nombre
0,Classic,14579
1,Supreme,11777
2,Chicken,10815
3,Veggie,11449


In [12]:
%%sql
-- Les trois top vente par catrgorie
SELECT pizza_category ,pizza_name, SUM(total_price) AS CA ,
            DENSE_RANK()
            OVER(PARTITION BY pizza_category ORDER BY SUM(total_price)DESC) AS Rang
FROM pizza_sales
GROUP BY pizza_category ,pizza_name 
LIMIT 3;

,pizza_category,pizza_name,CA,Rang
0,Supreme,The Spicy Italian Pizza,34831.25,1
1,Supreme,The Italian Supreme Pizza,33476.75,2
2,Supreme,The Sicilian Pizza,30940.50,3


In [13]:
%%sql
-- LES TOP 3
SELECT pizza_category ,pizza_name,CA,Rang
FROM  (SELECT pizza_category ,pizza_name, SUM(total_price) AS CA ,
            DENSE_RANK()
            OVER(PARTITION BY pizza_category ORDER BY SUM(total_price)DESC) AS Rang
FROM pizza_sales
GROUP BY pizza_category ,pizza_name ) AS Tem
WHERE Rang BETWEEN 1 AND 3;

,pizza_category,pizza_name,CA,Rang
0,Veggie,The Four Cheese Pizza,32265.70,1
1,Veggie,The Mexicana Pizza,26780.75,2
2,Veggie,The Five Cheese Pizza,26066.50,3
3,Supreme,The Spicy Italian Pizza,34831.25,1
4,Supreme,The Italian Supreme Pizza,33476.75,2
5,Supreme,The Sicilian Pizza,30940.50,3
6,Chicken,The Thai Chicken Pizza,43434.25,1
7,Chicken,The Barbecue Chicken Pizza,42768.00,2
8,Chicken,The California Chicken Pizza,41409.50,3
9,Classic,The Classic Deluxe Pizza,38180.50,1


In [14]:
%%sql
-- avec le CTE
WITH CTE AS (
    SELECT pizza_category ,pizza_name, SUM(total_price) AS CA ,
            DENSE_RANK()
            OVER(PARTITION BY pizza_category ORDER BY SUM(total_price)DESC) AS Rang
FROM pizza_sales
GROUP BY pizza_category ,pizza_name )

SELECT pizza_category ,pizza_name,CA,Rang
from CTE
WHERE Rang BETWEEN 1 and 3


,pizza_category,pizza_name,CA,Rang
0,Supreme,The Spicy Italian Pizza,34831.25,1
1,Supreme,The Italian Supreme Pizza,33476.75,2
2,Supreme,The Sicilian Pizza,30940.50,3
3,Veggie,The Four Cheese Pizza,32265.70,1
4,Veggie,The Mexicana Pizza,26780.75,2
5,Veggie,The Five Cheese Pizza,26066.50,3
6,Chicken,The Thai Chicken Pizza,43434.25,1
7,Chicken,The Barbecue Chicken Pizza,42768.00,2
8,Chicken,The California Chicken Pizza,41409.50,3
9,Classic,The Classic Deluxe Pizza,38180.50,1


In [15]:
%%sql
-- Les  top 5 ventes par taille
SELECT pizza_size, pizza_name,pizza_category, SUM(total_price) AS CA ,
            DENSE_RANK()
            OVER(PARTITION BY pizza_size ORDER BY SUM(total_price)DESC) AS Rang
FROM pizza_sales
GROUP BY pizza_category ,pizza_name,pizza_size ;

,pizza_size,pizza_name,pizza_category,CA,Rang
0,XL,The Greek Pizza,Classic,14076.00,1
1,S,The Big Meat Pizza,Classic,22968.00,1
2,S,The Brie Carre Pizza,Supreme,11588.50,2
3,S,The Hawaiian Pizza,Classic,10710.00,3
4,S,The Classic Deluxe Pizza,Classic,9588.00,4
...,...,...,...,...,...
86,L,The Calabrese Pizza,Supreme,5589.00,26
87,L,The Greek Pizza,Classic,5227.50,27
88,L,The Italian Vegetables Pizza,Veggie,3990.00,28
89,L,The Chicken Alfredo Pizza,Chicken,3901.00,29


In [16]:
%%sql
 -- Les  top 5
 SELECT pizza_size, pizza_name,pizza_category,CA,Rang
 FROM (SELECT pizza_category ,pizza_name,pizza_size, SUM(total_price) AS CA ,
            DENSE_RANK()
            OVER(PARTITION BY pizza_size ORDER BY SUM(total_price)DESC) AS Rang
FROM pizza_sales
GROUP BY pizza_category ,pizza_name,pizza_size ) temp
WHERE Rang BETWEEN 1 AND 5;


,pizza_size,pizza_name,pizza_category,CA,Rang
0,XXL,The Greek Pizza,Classic,1006.60,1
1,XL,The Greek Pizza,Classic,14076.00,1
2,S,The Big Meat Pizza,Classic,22968.00,1
3,S,The Brie Carre Pizza,Supreme,11588.50,2
4,S,The Hawaiian Pizza,Classic,10710.00,3
5,S,The Classic Deluxe Pizza,Classic,9588.00,4
6,S,The Sicilian Pizza,Supreme,9199.75,5
7,M,The Classic Deluxe Pizza,Classic,18896.00,1
8,M,The Barbecue Chicken Pizza,Chicken,16013.00,2
9,M,The California Chicken Pizza,Chicken,15812.00,3


In [17]:
%%sql
-- LES 5 PIZZA LES MOINS VENDUS
SELECT pizza_name,pizza_category,SUM(total_price) AS Total
FROM pizza_sales
GROUP BY pizza_name,pizza_category
ORDER BY Total ASC
LIMIT 5;

,pizza_name,pizza_category,Total
0,The Brie Carre Pizza,Supreme,11588.50
1,The Green Garden Pizza,Veggie,13955.75
2,The Spinach Supreme Pizza,Supreme,15277.75
3,The Mediterranean Pizza,Veggie,15360.50
4,The Spinach Pesto Pizza,Veggie,15596.00


In [18]:
%%sql
-- LES 3 PIZZA LES PLUS VENDUS
SELECT pizza_name,pizza_category,SUM(total_price) AS Total
FROM pizza_sales
GROUP BY pizza_name,pizza_category
ORDER BY Total DESC
LIMIT 5

,pizza_name,pizza_category,Total
0,The Thai Chicken Pizza,Chicken,43434.25
1,The Barbecue Chicken Pizza,Chicken,42768.00
2,The California Chicken Pizza,Chicken,41409.50
3,The Classic Deluxe Pizza,Classic,38180.50
4,The Spicy Italian Pizza,Supreme,34831.25


In [ ]:
%%sql
-- Le revenu total des ventes
SELECT SUM(total_price) AS Revenue
FROM pizza_sales


,Revenue
0,817860.05


In [20]:
%%sql
-- ROLLBACK
-- total vente au ée semetre  de l,année 2022
SELECT SUM(total_price) AS Revenue
FROM pizza_sales

WHERE order_date BETWEEN  '2022-06-01' AND  '2022-12-31'


,Revenue
0,472370.5


In [21]:
%%sql
ROLLBACK ;
-- total vente au deuxieme  semetre  de l'année 2022
SELECT SUM(total_price) AS Revenue
FROM pizza_sales

WHERE order_date NOT BETWEEN  '2022-07-01' AND  '2022-12-31'

,Revenue
0,413719.75


In [22]:
%%sql
ROLLBACK;
-- total vente au premier  semetre  de l'année 2022 par  categorie
SELECT pizza_category,SUM(total_price) AS Revenue
FROM pizza_sales

WHERE order_date BETWEEN  '2022-07-01' AND  '2022-12-31'
GROUP BY pizza_category

,pizza_category,Revenue
0,Chicken,97325.25
1,Classic,109705.20
2,Supreme,102172.30
3,Veggie,94937.55


In [23]:
%%sql
-- ROLLBACK
-- Total vente au Premiere  Trimestre  de lannée 2022
SELECT SUM(total_price) AS Revenue
FROM pizza_sales

WHERE order_date BETWEEN  '2022-01-01' AND  '2022-03-31'

,Revenue
0,205350.0


In [24]:
%%sql
-- ROLLBACK
-- Total vente au Premiere  Trimestre  de lannée 2022
SELECT pizza_size,SUM(total_price) AS Revenue
FROM pizza_sales
WHERE order_date BETWEEN  '2022-01-01' AND  '2022-03-31'
GROUP BY pizza_size
ORDER BY Revenue DESC

,pizza_size,Revenue
0,L,95229.65
1,M,61159.00
2,S,45384.25
3,XL,3289.50
4,XXL,287.60
